In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure plots render inline inside Jupyter
%matplotlib inline 

from market_data.yf_fred_market_data import MarketDataManager
from market_modelling.residual_boostrap import VARResidualBootstrapSimulator
from portfolio_models.linear_models import LongSPYWithTreasuryLadders

In [ ]:
data_manager = MarketDataManager(cache_filepath="market_data.parquet")
market_levels, market_returns = data_manager.get_aligned_data(force_refresh=False)
latest_row = market_levels.iloc[-1]
last_spx = latest_row["spx_close"]
last_yield3m = latest_row["yield_3m"]
last_yield5y = latest_row["yield_5y"]

simulator = VARResidualBootstrapSimulator(last_spx, last_yield3m, last_yield5y)
simulator.fit(returns_data=market_returns, levels_data=market_levels)

spx_path, yield3m_path, yield5y_path = simulator.simulate_paths(17640, 1)

strategy = LongSPYWithTreasuryLadders(0.9, 0.1, 500_000)
return_path = strategy.run_simulation(
    spx=spx_path[0, :],
    yield3m=yield3m_path[0, :],
    yield5y=yield5y_path[0, :],
    initial_nav = 8_000_000,
    days=17640,
    full_book=True)
book = strategy.transaction_book()

   

In [ ]:
spx_trajectory = spx_path[0, :] / 10.0
spx_df = pd.Series(spx_trajectory)
ema30 = spx_df.ewm(span=20, adjust=False).mean().values
sma90 = spx_df.rolling(window=63).mean()
sma180 = spx_df.rolling(window=126).mean() 

df = pd.DataFrame({
    "Day": range(0, len(spx_trajectory)),
    "SPX": list(spx_trajectory),
    "EMA30": ema30,
    "SMA90": sma90,
    "SMA180": sma180,
    "NAV": return_path
})
df.set_index("Day", inplace=True)

# 2. Set up the primary plot
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot the first time series (Left Y-axis)
color_price = "tab:blue"
ax1.set_xlabel("Day", fontsize=12)
ax1.set_ylabel("SPX Price ($)", color=color_price, fontsize=12)
line1 = ax1.plot(df.index, df["SPX"], color=color_price, linewidth=2, label="SPX Price ($)")
line2 = ax1.plot(df.index, df["EMA30"], color="tab:purple", linewidth=2, label="SPX EMA-30")
line3 = ax1.plot(df.index, df["SMA90"], color="tab:green", linewidth=2, label="SPX SMA-90")
line4 = ax1.plot(df.index, df["SMA180"], color="tab:orange", linewidth=2, label="SPX SMA-180")
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.grid(True, linestyle="--", alpha=0.3)

# 3. Create a twin axis sharing the same X-axis
ax2 = ax1.twinx()  

# Plot the second time series (Right Y-axis)
color_volume = "tab:red"
ax2.set_ylabel("Portfolio NAV", color=color_volume, fontsize=12)
line4 = ax2.plot(df.index, df["NAV"], color=color_volume, linewidth=2, label="NAV")
ax2.tick_params(axis='y', labelcolor=color_volume)

# 4. Combine legends from both axes into a single box
lines = line1 + line2 + line3 + line4
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")

# Title and formatting
plt.title("Simulated SPX vs Portfolio", fontsize=14, fontweight="bold")
fig.autofmt_xdate() # Auto-rotate date labels
plt.tight_layout()

plt.show()

# Massage the trade book to make it more presentable
sanitized_book = []
for i, row in enumerate(book):
    entry = book[i]
    symbol = ""
    position_size = ""
    rate = ""
    maturity = ""
    total = ""
    if "symbol" in entry:
        symbol = entry["symbol"]
    if "size" in entry:
        position_size = entry["size"]
    if "rate" in entry:
        rate = f"{entry["rate"]*100.0:.2f}%"
    if "maturity" in entry:
        maturity = f"{entry["maturity"]:.0f}"
    if "total" in entry:
        total = f"{entry["total"]:,.2f}"
        
    book_entry = {
        "Day": entry["day"] + 1,
        "NAV": f"{return_path[entry["day"]]:,.2f}",
        "SPY": f"{spx_trajectory[entry["day"]]:,.2f}",
        "Trade": entry["trade"].title(),
        "Symbol": symbol,
        "Price": f"{entry["price"]:,.2f}",
        "Position Size": position_size,
        "Total": total,
        "Description": entry["description"],
        "Rate": rate,
        "Maturity": maturity
    }

    sanitized_book.append(book_entry)
    

trading_book = pd.DataFrame(sanitized_book)
# Tell pandas to show all rows
#pd.set_option('display.max_rows', None)
# Tell pandas to show all columns (just in case they are hidden too)
#pd.set_option('display.max_columns', None)

In [ ]:
trading_book